# Demand Forecasting
**RetailPulse | Business Analytics + MLOps**

Time-series forecasting of daily order volumes using feature engineering and gradient boosting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import psycopg2

conn = psycopg2.connect(
    host='postgres', dbname='retailpulse',
    user='retailpulse', password='retailpulse123'
)

In [ ]:
# Load daily aggregated revenue data
df = pd.read_sql("""
    SELECT order_date, SUM(total_orders) AS total_orders,
           SUM(gross_revenue) AS gross_revenue,
           AVG(avg_order_value) AS avg_order_value,
           AVG(cancellation_rate_pct) AS cancellation_rate_pct
    FROM gold.agg_daily_revenue
    GROUP BY order_date
    ORDER BY order_date
""", conn)

df['order_date'] = pd.to_datetime(df['order_date'])
df = df.set_index('order_date').fillna(0)
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'Total days: {len(df)}')
df.head()

In [ ]:
# Feature engineering
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['lag_1']  = df['total_orders'].shift(1)
df['lag_7']  = df['total_orders'].shift(7)
df['lag_14'] = df['total_orders'].shift(14)
df['rolling_7']  = df['total_orders'].rolling(7).mean()
df['rolling_30'] = df['total_orders'].rolling(30).mean()

df = df.dropna()
print(f'Training rows after lag features: {len(df)}')

In [ ]:
FEATURES = ['day_of_week', 'month', 'quarter', 'is_weekend',
            'lag_1', 'lag_7', 'lag_14', 'rolling_7', 'rolling_30',
            'avg_order_value', 'cancellation_rate_pct']
TARGET = 'total_orders'

X = df[FEATURES]
y = df[TARGET]

# Time-series split (no data leakage)
tscv = TimeSeriesSplit(n_splits=3)
maes, r2s = [], []

for train_idx, test_idx in tscv.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    model = GradientBoostingRegressor(n_estimators=100, max_depth=4,
                                       learning_rate=0.05, random_state=42)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    maes.append(mean_absolute_error(y_te, preds))
    r2s.append(r2_score(y_te, preds))

print(f'Cross-val MAE: {np.mean(maes):.2f} ± {np.std(maes):.2f}')
print(f'Cross-val R²:  {np.mean(r2s):.4f} ± {np.std(r2s):.4f}')

In [ ]:
# Final model on full data, predict next 14 days
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance
importances.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Actual vs predicted (last 30 days)
last_30 = df.tail(30)
preds_30 = model.predict(last_30[FEATURES])
axes[1].plot(last_30.index, last_30[TARGET].values, label='Actual', color='steelblue', linewidth=2)
axes[1].plot(last_30.index, preds_30, label='Predicted', color='#E24A33',
             linestyle='--', linewidth=2)
axes[1].set_title('Actual vs Predicted Orders (Last 30 Days)', fontweight='bold')
axes[1].set_ylabel('Daily Orders')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('demand_forecast.png', dpi=150)
plt.show()

## Business Value

- **Inventory Planning**: Know expected demand 14 days ahead → reduce stockouts
- **Staffing**: Scale warehouse ops on predicted high-demand days
- **Marketing Budget**: Pre-allocate ad spend to high-demand periods
- **Supplier Negotiation**: Use forecasts to negotiate bulk purchase discounts